# 02 — LST (Land Surface Temperature) Kompoziti

Landsat 8/9 Collection 2 Level-2'den **2020-2024 yaz medyan kompoziti** (Haziran-Ağustos, sahne bulutu < %10), 30 m grid'e zonal stats ile bağlanır.

## Önkoşullar

1. **Google Earth Engine hesabı** — https://earthengine.google.com/ → register.
2. **Cloud project** — https://console.cloud.google.com/ → yeni proje (örn. `ee-ercangpg`). EE access'i etkinleştir.
3. **PowerShell'de project ID set et** (kalıcı):
   ```powershell
   [Environment]::SetEnvironmentVariable('GEE_PROJECT', 'ee-ercangpg', 'User')
   ```
   VS Code'u kapatıp yeniden açın (env variable yenilensin).
4. **İlk çalıştırma**: bir alttaki hücredeki `ee.Authenticate()` browser açacak — Google ile giriş yapın, token cache'lenecek.

## Çıktılar

- `data/processed/lst_summer_median_2020_2024.tif`
- `data/processed/grid_30m_lst.gpkg` — 30 m grid + LST stats
- `figures/02_lst_map.png`, `figures/02_lst_histogram.png`
- `tables/02_lst_summary.csv`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_GRID, DATA_PROCESSED, FIGURES, TABLES,
    CRS_PROJECTED, LST_YEARS, LST_MONTHS, CLOUD_COVER_THRESHOLD,
    GEE_PROJECT, LST_RASTER, LST_GRID_30M,
)
from src.gee_utils import (
    init_ee, summer_median_lst, boundary_to_ee_geometry,
)
from src.variables import zonal_stats_to_grid

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Earth Engine auth + init ---
import ee

# İlk çalıştırmada bunu UNCOMMENT edip çalıştırın (browser açılır):
ee.Authenticate()

project_id = GEE_PROJECT or os.environ.get("GEE_PROJECT")
if not project_id:
    raise RuntimeError(
        "GEE_PROJECT env variable yok!\n\n"
        "PowerShell'de kalıcı set edin (admin gerekmez):\n"
        "  [Environment]::SetEnvironmentVariable('GEE_PROJECT', 'ee-XXX', 'User')\n"
        "VS Code'u kapatıp yeniden açın, sonra bu hücreyi tekrar çalıştırın.\n\n"
        "Veya tek seferlik (sadece bu Python session'ı için):\n"
        "  os.environ['GEE_PROJECT'] = 'ee-XXX'"
    )

init_ee(project_id)
print(f"EE initialized — project: {project_id}")

In [ ]:
# --- Pilot sınırı yükle ve EE Geometry'e çevir ---
boundary_path = DATA_GRID / "pilot_boundary.gpkg"
if not boundary_path.exists():
    raise FileNotFoundError(
        f"{boundary_path} yok. Önce 01_grid_setup.ipynb'yi çalıştırın."
    )

boundary = gpd.read_file(boundary_path)
region_ee = boundary_to_ee_geometry(boundary)

area_km2 = boundary.to_crs(CRS_PROJECTED).area.iloc[0] / 1e6
print(f"Pilot alan: {area_km2:.2f} km²")
print(f"EE Geometry tipi: {region_ee.type().getInfo()}")

In [ ]:
# --- Yaz medyan LST kompoziti (2020-2024 Haziran-Ağustos, bulut < %10) ---
from src.gee_utils import landsat_lst_collection

coll = landsat_lst_collection(
    region=region_ee,
    years=LST_YEARS,
    months=LST_MONTHS,
    cloud_cover_max=CLOUD_COVER_THRESHOLD,
)
n_scenes = coll.size().getInfo()
print(f"Filtrelenen sahne sayısı: {n_scenes}")
if n_scenes == 0:
    raise RuntimeError(
        "Filtreden geçen sahne yok — bulut eşiğini gevşetmeyi deneyin "
        "(src/config.py: CLOUD_COVER_THRESHOLD)"
    )

lst_image = summer_median_lst(
    region=region_ee,
    years=LST_YEARS,
    months=LST_MONTHS,
    cloud_cover_max=CLOUD_COVER_THRESHOLD,
)
print(f"Bant: {lst_image.bandNames().getInfo()}")

In [ ]:
# --- GeoTIFF olarak indir ---
import geemap

if LST_RASTER.exists():
    print(f"Önceki indirme bulundu, atlandı: {LST_RASTER}")
    print("Yeniden indirmek istiyorsanız dosyayı silin ve hücreyi tekrar çalıştırın.")
else:
    print(f"İndiriliyor → {LST_RASTER}")
    geemap.ee_export_image(
        ee_object=lst_image,
        filename=str(LST_RASTER),
        scale=30,
        region=region_ee,
        file_per_band=False,
        crs="EPSG:32636",
    )
    print(f"Yazıldı: {LST_RASTER.stat().st_size/1024:.1f} KB")

In [ ]:
# --- Raster'ı yerel olarak aç ve haritada göster ---
import rasterio
from rasterio.plot import show as rio_show

with rasterio.open(LST_RASTER) as src:
    print(f"Boyut: {src.width} x {src.height}")
    print(f"CRS: {src.crs}")
    print(f"Bant sayısı: {src.count}")
    print(f"NoData: {src.nodata}")
    arr = src.read(1, masked=True)
    transform = src.transform

print(f"\nLST istatistikleri (°C):")
print(f"  min:    {arr.min():.2f}")
print(f"  max:    {arr.max():.2f}")
print(f"  mean:   {arr.mean():.2f}")
print(f"  median: {np.ma.median(arr):.2f}")

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(
    arr,
    extent=(transform[2], transform[2] + arr.shape[1] * transform[0],
            transform[5] + arr.shape[0] * transform[4], transform[5]),
    cmap="inferno", vmin=np.percentile(arr.compressed(), 2),
    vmax=np.percentile(arr.compressed(), 98),
)
boundary.to_crs(CRS_PROJECTED).boundary.plot(
    ax=ax, color="#0a9396", linewidth=1.5,
)
plt.colorbar(im, ax=ax, label="LST (°C)")
ax.set_title(f"Landsat 8/9 yaz medyan LST — {min(LST_YEARS)}-{max(LST_YEARS)}")
ax.set_aspect("equal")
ax.ticklabel_format(style="plain")
plt.tight_layout()
plt.savefig(FIGURES / "02_lst_map.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- 30m grid'e zonal stats ---
grid_30 = gpd.read_file(DATA_GRID / "grid_30m.gpkg")
print(f"30m grid: {len(grid_30):,} hücre")

grid_lst = zonal_stats_to_grid(
    grid=grid_30,
    raster_path=LST_RASTER,
    stats=("mean", "median", "std", "count"),
    prefix="lst_",
    all_touched=False,
)

n_with_data = grid_lst["lst_count"].fillna(0).gt(0).sum()
print(f"LST verisi olan hücre: {n_with_data:,} / {len(grid_lst):,}")
print(f"\nlst_mean özet:")
print(grid_lst["lst_mean"].describe().round(2))

In [ ]:
# --- LST histogram + grid haritası ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram
vals = grid_lst["lst_mean"].dropna()
axes[0].hist(vals, bins=50, color="#E76F51", edgecolor="black", alpha=0.85)
axes[0].axvline(vals.median(), color="#264653", linestyle="--",
                label=f"medyan: {vals.median():.1f}°C")
axes[0].set_xlabel("30 m hücre LST ortalaması (°C)")
axes[0].set_ylabel("Hücre sayısı")
axes[0].set_title("30m grid LST dağılımı")
axes[0].legend()

# Choropleth grid
grid_lst.plot(
    column="lst_mean", cmap="inferno",
    legend=True, legend_kwds={"label": "LST ortalama (°C)"},
    ax=axes[1], linewidth=0,
    vmin=np.percentile(vals, 2), vmax=np.percentile(vals, 98),
)
boundary.to_crs(CRS_PROJECTED).boundary.plot(
    ax=axes[1], color="#0a9396", linewidth=1,
)
axes[1].set_title("30m grid choropleth (LST mean)")
axes[1].set_aspect("equal")
axes[1].ticklabel_format(style="plain")

plt.tight_layout()
plt.savefig(FIGURES / "02_lst_histogram.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Kaydet + özet ---
grid_lst.to_file(LST_GRID_30M, driver="GPKG")
print(f"Kaydedildi: {LST_GRID_30M} ({LST_GRID_30M.stat().st_size/1024:.1f} KB)")

summary = pd.DataFrame({
    "metrik": [
        "sahne_sayisi", "yil_araligi", "aylar", "bulut_esigi_pct",
        "raster_genislik", "raster_yukseklik",
        "grid_30m_hucre", "lst_verili_hucre",
        "lst_mean_min", "lst_mean_max", "lst_mean_median",
        "lst_std_median",
    ],
    "deger": [
        n_scenes,
        f"{min(LST_YEARS)}-{max(LST_YEARS)}",
        ",".join(map(str, LST_MONTHS)),
        CLOUD_COVER_THRESHOLD,
        arr.shape[1], arr.shape[0],
        len(grid_lst), int(n_with_data),
        round(float(vals.min()), 2),
        round(float(vals.max()), 2),
        round(float(vals.median()), 2),
        round(float(grid_lst["lst_std"].median()), 2),
    ],
})
summary.to_csv(TABLES / "02_lst_summary.csv", index=False, encoding="utf-8-sig")
summary

## Özet

- Landsat 8/9 Collection 2 Level-2 yaz medyan LST kompoziti üretildi.
- QA_PIXEL bit-mask ile bulut/gölge/kar elendi; sahne-seviyesi `CLOUD_COVER < %10`.
- ST_B10 → Kelvin (×0.00341802 + 149.0) → Celsius dönüşümü.
- 30 m grid'e `lst_mean`, `lst_median`, `lst_std`, `lst_count` zonal stat olarak bağlandı.

## Sonraki Adım

`03_variables.ipynb` — Sentinel-2'den NDVI ve Albedo (Liang 2001), ESA WorldCover'dan geçirimsiz yüzey oranı; aynı 30 m grid'e bağlanacak.